# STAGE 13A — SAMPLE RETENTION AND NATIONAL PREDICTOR READINESS AUDIT

**Malaysia-only FYP — Pre-mapping quality gate**

This notebook **does not retrain models or modify previously reported performance**. It investigates whether excluded landslide cases are potentially recoverable, audits missing feature information and checks whether existing source paths are ready for a Malaysia-wide prediction surface. **Stage 13B mapping follows only after these results are reviewed.**

Preserve 1:2 case-control design, original nested spatial evaluation and saved Stage-12 model schemas. Do not automatically add recovered cases to a trained model.

# PART A — PROJECT PATHS AND INPUT FILES
## 1. Libraries and paths

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"
RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"

STATIC_MASTER_PATH = DEV_DIR / "03_derived_features" / "static_features_final" / "static_feature_samples_final.csv"
STATIC_MODEL_PATH = DEV_DIR / "04_model_ready" / "static_model_1to2.csv"
TEMPORAL_ANNOTATED_PATH = DEV_DIR / "03_derived_features" / "temporal_features_final" / "temporal_feature_samples_final_annotated.csv"
TEMPORAL_DIR = DEV_DIR / "03_derived_features" / "temporal_features_final"
POSITIVE_INVENTORY_PATH = DEV_DIR / "02_cleaned_inventory" / "malaysia_positive_inventory_final.csv"
ELIGIBLE_INVENTORY_PATH = DEV_DIR / "02_cleaned_inventory" / "malaysia_positive_inventory_ml_eligible.csv"
STAGE04_PATH = DEV_DIR / "01_preprocessing" / "stage04_revised_outputs" / "positive_inventory_with_ml_eligibility.csv"
STAGE12_DIR = DEV_DIR / "05_model_results" / "stage12_production_models"

OUTPUT_DIR = DEV_DIR / "05_model_results" / "stage13a_sample_coverage_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPORAL_FILES = {
    "rainfall": TEMPORAL_DIR / "temporal_model_rainfall.csv",
    "rainfall_soilmoisture": TEMPORAL_DIR / "temporal_model_rainfall_soilmoisture.csv",
    "landcover": TEMPORAL_DIR / "temporal_model_landcover.csv",
    "full_enhanced": TEMPORAL_DIR / "temporal_model_full_enhanced.csv",
}

INPUT_PATHS = {
    "reviewed_inventory": POSITIVE_INVENTORY_PATH,
    "eligible_inventory": ELIGIBLE_INVENTORY_PATH,
    "stage04_eligibility_audit": STAGE04_PATH,
    "static_master": STATIC_MASTER_PATH,
    "static_1to2": STATIC_MODEL_PATH,
    "temporal_annotated": TEMPORAL_ANNOTATED_PATH,
    **{f"temporal_{key}": value for key, value in TEMPORAL_FILES.items()},
    "static_schema": STAGE12_DIR / "static_feature_schema.json",
    "temporal_schema": STAGE12_DIR / "temporal_feature_schema.json",
    "static_model": STAGE12_DIR / "static_rf_pipeline.joblib",
    "temporal_model": STAGE12_DIR / "temporal_rf_pipeline.joblib",
}

inventory_paths = pd.DataFrame([
    {"source": name, "exists": path.exists(), "path": str(path)}
    for name, path in INPUT_PATHS.items()
])

display(inventory_paths)

required = ["static_master", "static_1to2", "temporal_full_enhanced"]
missing_required = inventory_paths.loc[
    inventory_paths["source"].isin(required) & ~inventory_paths["exists"],
    "source"
].tolist()
if missing_required:
    raise FileNotFoundError(f"Missing required files: {missing_required}. Confirm PROJECT_ROOT and original folder structure.")

inventory_paths.to_csv(OUTPUT_DIR / "input_file_path_audit.csv", index=False)


,source,exists,path
0,reviewed_inventory,True,C:\Landslide_FYP\Landslide_FYP_Development\02_...
1,eligible_inventory,True,C:\Landslide_FYP\Landslide_FYP_Development\02_...
2,stage04_eligibility_audit,True,C:\Landslide_FYP\Landslide_FYP_Development\01_...
3,static_master,True,C:\Landslide_FYP\Landslide_FYP_Development\03_...
4,static_1to2,True,C:\Landslide_FYP\Landslide_FYP_Development\04_...
5,temporal_annotated,True,C:\Landslide_FYP\Landslide_FYP_Development\03_...
6,temporal_rainfall,True,C:\Landslide_FYP\Landslide_FYP_Development\03_...
7,temporal_rainfall_soilmoisture,True,C:\Landslide_FYP\Landslide_FYP_Development\03_...
8,temporal_landcover,True,C:\Landslide_FYP\Landslide_FYP_Development\03_...
9,temporal_full_enhanced,True,C:\Landslide_FYP\Landslide_FYP_Development\03_...


## 2. Load exact model-ready sources without changing them

In [2]:
reviewed = pd.read_csv(POSITIVE_INVENTORY_PATH) if POSITIVE_INVENTORY_PATH.exists() else pd.DataFrame()
eligible = pd.read_csv(ELIGIBLE_INVENTORY_PATH) if ELIGIBLE_INVENTORY_PATH.exists() else pd.DataFrame()
stage04 = pd.read_csv(STAGE04_PATH) if STAGE04_PATH.exists() else pd.DataFrame()
static_master = pd.read_csv(STATIC_MASTER_PATH)
static_model = pd.read_csv(STATIC_MODEL_PATH)
temporal_annotated = pd.read_csv(TEMPORAL_ANNOTATED_PATH) if TEMPORAL_ANNOTATED_PATH.exists() else pd.DataFrame()
temporal = {
    name: pd.read_csv(path)
    for name, path in TEMPORAL_FILES.items()
    if path.exists()
}
with open(STAGE12_DIR / "static_feature_schema.json", encoding="utf-8") as f:
    static_schema = json.load(f)
with open(STAGE12_DIR / "temporal_feature_schema.json", encoding="utf-8") as f:
    temporal_schema = json.load(f)

for name, data in {
    "reviewed": reviewed, "eligible": eligible, "static_master": static_master,
    "static_model": static_model, "temporal_annotated": temporal_annotated,
    **{f"temporal_{k}": v for k, v in temporal.items()},
}.items():
    print(f"{name:30s} {len(data):6d} rows | {len(data.columns):3d} columns")


reviewed                          287 rows |  19 columns
eligible                          200 rows |  29 columns
static_master                    3200 rows |  49 columns
static_model                      582 rows |  53 columns
temporal_annotated                567 rows |  92 columns
temporal_rainfall                 567 rows |  90 columns
temporal_rainfall_soilmoisture    474 rows |  91 columns
temporal_landcover                444 rows |  91 columns
temporal_full_enhanced            363 rows |  92 columns


# PART B — RECONCILE SAMPLE COUNTS
## 3. Explain exactly which records enter each dataset

In [3]:
def static_positives(df):
    if "target" in df.columns:
        return df.loc[pd.to_numeric(df["target"], errors="coerce").eq(1)].copy()
    return df.loc[df["sample_class"].eq("landslide")].copy()


def temporal_cases(df):
    return df.loc[df["temporal_role"].eq("case")].copy()


static_cases = static_positives(static_model)

count_rows = [
    {"step": "Reviewed inventory", "cases": len(reviewed), "controls": 0},
    {"step": "Coordinate-eligible inventory", "cases": len(eligible), "controls": 0},
    {"step": "Final static 1:2 model", "cases": len(static_cases), "controls": len(static_model) - len(static_cases)},
]
for name, data in temporal.items():
    count_rows.append({
        "step": f"Temporal {name}",
        "cases": len(temporal_cases(data)),
        "controls": len(data) - len(temporal_cases(data)),
    })

retention = pd.DataFrame(count_rows)
retention["total"] = retention["cases"] + retention["controls"]
retention["retained_pct_of_reviewed"] = np.where(
    len(reviewed) > 0, 100 * retention["cases"] / len(reviewed), np.nan
)
retention["ratio_check_1to2"] = np.where(
    retention["controls"] > 0,
    retention["controls"] == 2 * retention["cases"],
    np.nan
)

retention.to_csv(OUTPUT_DIR / "sample_retention_funnel.csv", index=False)
display(retention)

expected = {"Reviewed inventory":287,"Coordinate-eligible inventory":200,"Final static 1:2 model":194,"Temporal full_enhanced":121}
for stage, number in expected.items():
    rows = retention.loc[retention["step"].eq(stage)]
    if len(rows) and int(rows.iloc[0]["cases"]) != number:
        print("WARNING: Historical count differs:", stage, number, "!=", int(rows.iloc[0]["cases"]))


,step,cases,controls,total,retained_pct_of_reviewed,ratio_check_1to2
0,Reviewed inventory,287,0,287,100.000000,NaN
1,Coordinate-eligible inventory,200,0,200,69.686411,NaN
2,Final static 1:2 model,194,388,582,67.595819,1.0
3,Temporal rainfall,189,378,567,65.853659,1.0
4,Temporal rainfall_soilmoisture,158,316,474,55.052265,1.0
5,Temporal landcover,148,296,444,51.567944,1.0
6,Temporal full_enhanced,121,242,363,42.160279,1.0


## 4. Audit the Stage 04 coordinate eligibility reasons and static exclusions

In [4]:
eligibility_columns = [c for c in stage04.columns if any(k in c.lower() for k in ["eligib", "precision", "accuracy", "exclu", "reason", "coordinate", "source", "state"])]
print("Eligibility audit columns:", eligibility_columns)

reason_rows = []
for c in eligibility_columns:
    if any(k in c.lower() for k in ["reason", "status", "eligib", "precision", "accuracy", "exclu"]):
        counts = stage04[c].fillna("<MISSING>").astype(str).value_counts(dropna=False)
        for value, cnt in counts.head(40).items():
            reason_rows.append({"source": "stage04", "column": c, "value": value, "count": cnt})

if "sample_class" in static_master.columns:
    static_positive_master = static_master.loc[static_master["sample_class"].eq("landslide")].copy()
else:
    static_positive_master = static_positives(static_master)

for c in static_positive_master.columns:
    if any(k in c.lower() for k in ["valid", "reason", "soil", "fault", "missing"]):
        if static_positive_master[c].dtype == "object" or static_positive_master[c].nunique(dropna=False) <= 10:
            for value, cnt in static_positive_master[c].fillna("<MISSING>").astype(str).value_counts().head(20).items():
                reason_rows.append({"source": "static_master", "column": c, "value": value, "count": cnt})

reason_table = pd.DataFrame(reason_rows, columns=["source","column","value","count"])
reason_table.to_csv(OUTPUT_DIR / "exclusion_reason_distributions.csv", index=False)
display(reason_table.head(80))


Eligibility audit columns: ['source_dataset', 'source_record_id', 'source_event_id', 'event_date_source_reliable_for_review', 'state_or_admin', 'location_accuracy_source', 'duplicate_reason', 'location_accuracy_m', 'ml_eligible', 'ml_exclusion_reason', 'date_precision_for_modelling', 'annual_landcover_eligible', 'antecedent_rainfall_eligible']


,source,column,value,count
0,stage04,location_accuracy_source,5km,66
1,stage04,location_accuracy_source,1km,61
2,stage04,location_accuracy_source,<MISSING>,52
3,stage04,location_accuracy_source,25km,34
4,stage04,location_accuracy_source,10km,31
...,...,...,...,...
66,static_master,soil_valid_final,False,6
67,static_master,national_core_static_valid_final,True,194
68,static_master,national_core_static_valid_final,False,6
69,static_master,geology_enhanced_valid_final,True,192


# PART C — FEATURE COMPLETENESS AND RECOVERY OPPORTUNITIES
## 5. Identify exactly which temporal inputs are unavailable

In [5]:
static_features = static_schema["feature_order"]
temporal_features = temporal_schema["feature_order"]

feature_rows = []
for name, df, feats in [
    ("static_model", static_model, static_features),
    ("temporal_annotated", temporal_annotated, temporal_features),
    *[(f"temporal_{key}", data, temporal_features) for key, data in temporal.items()],
]:
    for group, subset in [("all", df)] + (
        [("case", temporal_cases(df)), ("control", df.loc[df["temporal_role"].ne("case")])]
        if "temporal_role" in df.columns else []
    ):
        for feat in feats:
            exists = feat in subset.columns
            missing = int(subset[feat].isna().sum()) if exists else len(subset)
            feature_rows.append({
                "dataset": name, "role": group, "feature": feat,
                "column_exists": exists, "rows": len(subset),
                "missing_rows": missing,
                "missing_pct": round(100 * missing / len(subset), 2) if len(subset) else np.nan,
            })

features = pd.DataFrame(feature_rows)
features.to_csv(OUTPUT_DIR / "feature_completeness_by_subset.csv", index=False)
display(features.loc[(features["role"].eq("case")) & (features["missing_rows"] > 0)].sort_values(["dataset","missing_rows"], ascending=[True,False]).head(100))

print("Note: imputed values allow training but are not equivalent to historically observed data.")


,dataset,role,feature,column_exists,rows,missing_rows,missing_pct
47,temporal_annotated,case,geology_age,True,189,34,17.99
44,temporal_annotated,case,soil_moisture_0_7cm_7d_mean,True,189,17,8.99
45,temporal_annotated,case,soil_moisture_7_28cm_7d_mean,True,189,17,8.99
49,temporal_annotated,case,landcover_harmonised,True,189,15,7.94
48,temporal_annotated,case,geology_unit_type,True,189,2,1.06
251,temporal_full_enhanced,case,geology_age,True,121,22,18.18
200,temporal_landcover,case,geology_age,True,148,28,18.92
197,temporal_landcover,case,soil_moisture_0_7cm_7d_mean,True,148,13,8.78
198,temporal_landcover,case,soil_moisture_7_28cm_7d_mean,True,148,13,8.78
201,temporal_landcover,case,geology_unit_type,True,148,1,0.68


Note: imputed values allow training but are not equivalent to historically observed data.


## 6. Case-matching audit — do not silently join duplicate coordinates

In [6]:
# The most reliable matching key is a persistent landslide/sample identifier.
# We never assume lat/lon is unique (multiple events may share a location).

candidate_ids = [
    "matched_positive_id", "positive_id", "sample_id", "landslide_id", "event_id",
    "inventory_id", "record_id", "source_id", "case_id",
]

case_tables = {
    "reviewed": reviewed,
    "eligible": eligible,
    "static": static_cases,
    **{f"temporal_{k}": temporal_cases(v) for k,v in temporal.items()},
}

key_rows = []
for name, df in case_tables.items():
    for c in candidate_ids:
        if c in df.columns:
            key_rows.append({
                "dataset": name, "column": c,
                "rows": len(df),
                "non_null": int(df[c].notna().sum()),
                "distinct": int(df[c].nunique(dropna=True)),
                "duplicates": int(df.loc[df[c].notna(), c].duplicated().sum()),
            })

key_audit = pd.DataFrame(key_rows, columns=["dataset","column","rows","non_null","distinct","duplicates"])
key_audit.to_csv(OUTPUT_DIR / "case_identifier_audit.csv", index=False)
display(key_audit)

# Find a shared verified identifier for static versus temporal cases.
full_cases = case_tables.get("temporal_full_enhanced", pd.DataFrame())
paired_options = []
for s in candidate_ids:
    if s not in static_cases.columns:
        continue
    static_values = static_cases[s].dropna().astype(str)
    if static_values.duplicated().any():
        continue
    for t in candidate_ids:
        if t not in full_cases.columns:
            continue
        temporal_values = full_cases[t].dropna().astype(str)
        if temporal_values.duplicated().any():
            continue
        overlap = len(set(static_values).intersection(set(temporal_values)))
        paired_options.append({"static_key":s,"temporal_key":t,"matched_unique":overlap,"temporal_cases":len(full_cases)})

key_pairs = pd.DataFrame(paired_options, columns=["static_key","temporal_key","matched_unique","temporal_cases"])
key_pairs.to_csv(OUTPUT_DIR / "case_identifier_crosswalk_candidates.csv", index=False)
display(key_pairs.sort_values("matched_unique", ascending=False).head(15))

if len(key_pairs) == 0 or key_pairs["matched_unique"].max() < len(full_cases):
    print("REVIEW REQUIRED: No fully verified case-ID crosswalk. Do NOT join by coordinate or infer causes for individual missing records.")
else:
    print("Candidate key overlap found. Verify semantics of matching columns before interpreting row-level exclusions.")


,dataset,column,rows,non_null,distinct,duplicates
0,reviewed,positive_id,287,287,287,0
1,eligible,positive_id,200,200,200,0
2,static,sample_id,194,194,194,0
3,temporal_rainfall,matched_positive_id,189,189,189,0
4,temporal_rainfall,sample_id,189,189,189,0
5,temporal_rainfall_soilmoisture,matched_positive_id,158,158,158,0
6,temporal_rainfall_soilmoisture,sample_id,158,158,158,0
7,temporal_landcover,matched_positive_id,148,148,148,0
8,temporal_landcover,sample_id,148,148,148,0
9,temporal_full_enhanced,matched_positive_id,121,121,121,0


,static_key,temporal_key,matched_unique,temporal_cases
0,sample_id,matched_positive_id,121,121
1,sample_id,sample_id,121,121


Candidate key overlap found. Verify semantics of matching columns before interpreting row-level exclusions.


## 7. Inspect individual missing features in the full temporal source

In [7]:
if len(temporal_annotated) and "temporal_role" in temporal_annotated.columns:
    source_cases = temporal_cases(temporal_annotated)
    raw_missing = pd.DataFrame({
        "feature": [c for c in temporal_features if c in source_cases.columns],
        "missing_cases": [int(source_cases[c].isna().sum()) for c in temporal_features if c in source_cases.columns],
        "total_cases": len(source_cases),
    })
    raw_missing["missing_pct"] = (100 * raw_missing["missing_cases"] / max(len(source_cases),1)).round(2)
    raw_missing.to_csv(OUTPUT_DIR / "temporal_source_missingness.csv", index=False)
    display(raw_missing.sort_values("missing_cases", ascending=False))
else:
    print("Annotated temporal source lacks temporal_role or file is unavailable. Need Stage 05B audited data for cause-level investigation.")


,feature,missing_cases,total_cases,missing_pct
14,geology_age,34,189,17.99
11,soil_moisture_0_7cm_7d_mean,17,189,8.99
12,soil_moisture_7_28cm_7d_mean,17,189,8.99
16,landcover_harmonised,15,189,7.94
15,geology_unit_type,2,189,1.06
0,elevation_m,0,189,0.00
1,slope_deg,0,189,0.00
3,soil_sand,0,189,0.00
2,soil_clay,0,189,0.00
8,rain_3d_mm,0,189,0.00


# PART D — NATIONAL PREDICTOR READINESS
## 8. Explicit coverage inventory; never claim national coverage from training-point completeness

In [8]:
raw_sources = {
    "Malaysia boundary": RAW_DIR / "02_Boundary" / "mys_admin0.geojson",
    "Soil clay": RAW_DIR / "03_Factors" / "Soil" / "Clay" / "soilgrids_clay_malaysia.tif",
    "Soil sand": RAW_DIR / "03_Factors" / "Soil" / "Sand" / "soilgrids_sand_malaysia.tif",
    "Soil silt": RAW_DIR / "03_Factors" / "Soil" / "Silt" / "soilgrids_silt_malaysia.tif",
    "Soil bulk density": RAW_DIR / "03_Factors" / "Soil" / "Bulk_Density" / "soilgrids_bdod_malaysia.tif",
    "Waterways": RAW_DIR / "03_Factors" / "Waterways" / "gis_osm_waterways_free_1.shp",
}

coverage_rows = [{"source":name,"path":str(path),"exists":path.exists(),"verified_national_coverage":False}
                 for name,path in raw_sources.items()]
# Existing source files are NOT proof of continuous valid raster coverage.
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(OUTPUT_DIR / "national_predictor_source_readiness.csv", index=False)
display(coverage)

print("Terrain SRTM, JMG national lithology and fault services still require explicit spatial coverage/raster QA in Stage 13B.")
print("Never replace missing geology ages with made-up categories or extrapolate detailed raster precision from coarser sources.")


,source,path,exists,verified_national_coverage
0,Malaysia boundary,C:\Landslide_FYP\Landslide_FYP_Data\02_Boundar...,True,False
1,Soil clay,C:\Landslide_FYP\Landslide_FYP_Data\03_Factors...,True,False
2,Soil sand,C:\Landslide_FYP\Landslide_FYP_Data\03_Factors...,True,False
3,Soil silt,C:\Landslide_FYP\Landslide_FYP_Data\03_Factors...,True,False
4,Soil bulk density,C:\Landslide_FYP\Landslide_FYP_Data\03_Factors...,True,False
5,Waterways,C:\Landslide_FYP\Landslide_FYP_Data\03_Factors...,True,False


Terrain SRTM, JMG national lithology and fault services still require explicit spatial coverage/raster QA in Stage 13B.
Never replace missing geology ages with made-up categories or extrapolate detailed raster precision from coarser sources.


# PART E — DECISION GATE
## 9. Recommendations and what to send for review

In [9]:
checks = [
    {"check":"Model-ready class ratios", "status":"AUDITED", "note":"See sample_retention_funnel.csv"},
    {"check":"Individual historical-case eligibility causes", "status":"REVIEW REQUIRED", "note":"Review stage04 and static reason distributions"},
    {"check":"Complete temporal source missingness", "status":"AUDITED IF SOURCE AVAILABLE", "note":"See temporal_source_missingness.csv"},
    {"check":"Static-to-temporal case identity matching", "status":"REVIEW REQUIRED", "note":"Require trusted ID crosswalk; do not deduplicate by coordinates"},
    {"check":"Nationwide static predictor surface", "status":"NOT YET VALIDATED", "note":"Requires aligned nationwide rasters, vector processing and no-data audit"},
    {"check":"Three-algorithm spatial benchmark and metrics", "status":"PRESERVED", "note":"Use Stage 08–11 held-out ROC-AUC, PR-AUC, balanced accuracy, precision, recall, F1 and confusion matrix"},
    {"check":"Production RF pipelines", "status":"PRESERVED", "note":"Do not retrain unless the model-ready dataset is formally revised; rerun all comparable validation if revised"},
]
readiness = pd.DataFrame(checks)
readiness.to_csv(OUTPUT_DIR / "stage13a_decision_gate.csv", index=False)
display(readiness)

print("\nSEND FOR REVIEW:")
for file in ["sample_retention_funnel.csv", "exclusion_reason_distributions.csv", "feature_completeness_by_subset.csv", "case_identifier_audit.csv", "case_identifier_crosswalk_candidates.csv", "temporal_source_missingness.csv", "national_predictor_source_readiness.csv", "stage13a_decision_gate.csv"]:
    print(" -", file)


,check,status,note
0,Model-ready class ratios,AUDITED,See sample_retention_funnel.csv
1,Individual historical-case eligibility causes,REVIEW REQUIRED,Review stage04 and static reason distributions
2,Complete temporal source missingness,AUDITED IF SOURCE AVAILABLE,See temporal_source_missingness.csv
3,Static-to-temporal case identity matching,REVIEW REQUIRED,Require trusted ID crosswalk; do not deduplica...
4,Nationwide static predictor surface,NOT YET VALIDATED,"Requires aligned nationwide rasters, vector pr..."
5,Three-algorithm spatial benchmark and metrics,PRESERVED,"Use Stage 08–11 held-out ROC-AUC, PR-AUC, bala..."
6,Production RF pipelines,PRESERVED,Do not retrain unless the model-ready dataset ...



SEND FOR REVIEW:
 - sample_retention_funnel.csv
 - exclusion_reason_distributions.csv
 - feature_completeness_by_subset.csv
 - case_identifier_audit.csv
 - case_identifier_crosswalk_candidates.csv
 - temporal_source_missingness.csv
 - national_predictor_source_readiness.csv
 - stage13a_decision_gate.csv


## Stage 13A interpretation

- **194 and 121 are positive cases**; 388 and 242 are selected pseudo-absences. More controls do not mean more documented historical landslides.
- Check Stage 04 coordinate exclusions, then the 6 lost during static QC, then the 73 lost at the full temporal intersection.
- Missing values, unmatched dates, source-year gaps, invalid environmental samples and absent vector coverage are **possible causes**, not proven reasons until linked to individual cases.
- If recovery changes the training population, compare LR/RF/XGBoost again on the **same revised cases**, same controls and spatial folds. Report all old metrics as historical benchmarks rather than directly comparing unlike splits.
- Proceed to **Stage 13B national predictor raster creation and susceptibility surface** after reviewing coverage and case-recovery evidence.
